# Transform Results Data

1. Read bronze `results` table
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
4. rename columns to make them more meaningful(date → race_date, grid → grid_position, laps → completede_laps, number → car_number, position → finish_position)
5. Filter out rows where `season`, `round`, `cunstructor_id` or `driver_id` is null (business key validation)
6. Remove duplicate records
7. Transform values of column `race_name` to Title Case
8. Write the transformed data to silver `results` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.results'
silver_table = f'{catalog_name}.{silver_schema}.results'

In [0]:
from pyspark.sql import functions as F

### Step 1 - Read bronze `results` table


In [0]:
results_df = spark.table(bronze_table)

In [0]:
display(results_df)

### Stpe 2 - Keep only the columns required for analytics (Drop `url` column)

In [0]:
results_dropped_df = results_df.drop(
    F.col("url")
)

### Step 3 & 4 - Standardise column names 
- Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
- rename columns to make them more meaningful(date → race_date, grid → grid_position, laps → completede_laps, number → car_number, position → finish_position)

In [0]:
results_renamed_df = results_dropped_df.withColumnsRenamed({
    'constructorId': 'constructor_id',
    "driverId": 'driver_id',
    'raceName':'race_name',
    'positionText' : 'finish_position_text',
    'date': 'race_date',
    'grid': 'grid_position',
    'laps':'completede_laps',
    'number':'car_number',
    'position': 'finish_position'
})

In [0]:
display(results_renamed_df)

### Step 5 - Filter out rows where `season`, `round`, `cunstructor_id` or `driver_id` is null (business key validation)

In [0]:
results_valid_df = (
     results_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
)

### Step 6 - Remove duplicates

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(['season', 'round', 'constructor_id', 'driver_id'])

### Step 7 - Transform values of column `nationality` to Title Case

In [0]:
results_final_df = (
    results_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

In [0]:
display(results_final_df)

### Step 8 - Write the transformed data to silver `constructors` table

In [0]:
(
    results_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))